In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

import torch
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [36]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
df.shape

(30006, 22)

In [37]:
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,main_image,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,None,Калитники,9.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,9.0,16.0
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,None,Соколиная гора,8.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,None,Тушинская,10.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,2.0,48.0
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,None,Бутырская,17.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,None,Коммунарка,14.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,5.0,16.0


In [40]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruBert-base")
model = AutoModel.from_pretrained("ai-forever/ruBert-base")

print(f"Total_params = {sum(p.numel() for p in model.parameters()) / 1024**3} GB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20779.39it/s]
BertModel LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total_params = 0.1660616397857666 GB


In [42]:
text = df['description'][0]
text

'Номер лота: 99696. Панорамный вид из больших окон на город. Тихо, дорога не оживленная. У дома отличная управляющая компания, на входе охрана, парковку чистят, во дворе чисто. Инфраструктура: Удобное расположение: до станции метро и МЦД-3'

In [43]:
t = tokenizer(text, padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    model_output = model(**{k: v.to(model.device) for k, v in t.items()})
embeddings = model_output.last_hidden_state[:, 0, :]
embeddings = torch.nn.functional.normalize(embeddings)

In [45]:
embeddings.shape

torch.Size([1, 768])

In [ ]:
def embed_bert_cls(text, model, tokenizer):
    t = tokenizer(text, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**{k: v.to(model.device) for k, v in t.items()})
    embeddings = model_output.last_hidden_state[:, 0, :]
    embeddings = torch.nn.functional.normalize(embeddings)
    return embeddings[0].cpu().numpy()

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 4422.34it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
for i, row in tqdm(df.iterrows()):
    print(i)
    break

0it [00:00, ?it/s]

0


In [59]:
row.metro

'Калитники'

In [60]:
embeds = []
for i, row in tqdm(df.iterrows()):
     metro = str(row['metro'])
     address = str(row['address'])
     author = str(row['author'])
     title = str(row['title'])
     description = str(row['description'])

     text = f"Данные обьявления о продаже квартиры \
     Метро: {metro} Адресс: {address} Автор: {author} Тип помешения: {title} Описание: {description}"

     emb = embed_bert_cls(text, model, tokenizer)
     embeds.append(emb)

30006it [38:07, 13.12it/s]


In [61]:
df['description_embedding'] = embeds

In [62]:
df.to_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/ya_realty_with_txt_embeds.parquet', index=False)